In [ ]:
from google.colab import drive
drive.mount('/content/drive')

**DATA COLLECTION** - IMPORTING TWITTER SENTIMENT DATASET

TweetEval Dataset

In [ ]:
from datasets import load_dataset
dataset = load_dataset("tweet_eval", "sentiment")
print(dataset)

Converting into Dataframe

In [ ]:
import pandas as pd

# Convert train split into DataFrame
df_train = dataset["train"].to_pandas()
df_val   = dataset["validation"].to_pandas()
df_test  = dataset["test"].to_pandas()


In [ ]:
print(df_train.head())

In [ ]:
twitter_data= df_train

In [ ]:
twitter_data.head()

**IMPORTING THE DEPENDENCIES (LIBRARIES AND MODULES)**

In [ ]:
!pip install wordcloud

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from wordcloud import WordCloud
import re   # regular expression (for pattern matching)
from nltk.corpus import stopwords  # nltk -> natural langage toolkit  ;  stopwords -> a, an, the, is, for, by, my, etc. (nltk=library , corpus=module)
from nltk.stem.porter import PorterStemmer  # Stemming -> reducing a word into it's root/key word
from sklearn.feature_extraction.text import TfidfVectorizer  # It changes the textual data into numerical data, so that we can feed it to the ML model
from sklearn.model_selection import train_test_split   # It splits entire data into training and testing data ; training data -> trains ML model, test data -> tests/evaluates the model
from sklearn.linear_model import LogisticRegression  # LR is the ML model that we our using
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

**EDA (EXPLORATORY DATA ANALYSIS)**

In [ ]:
# Printing the first 5 rows
twitter_data.head()

In [ ]:
# Reaming the columns and reading the dataset again

# rename 'label' -> 'target'
twitter_data = twitter_data.rename(columns={"label": "target"})

In [ ]:
# Again printing the first 5 rows
twitter_data.head()

In [ ]:
# printing any 8 rows
twitter_data.sample(8)

In [ ]:
# Printing the number of rows and columns of the dataset
twitter_data.shape

In [ ]:
# Printing the information of the data in the columns
twitter_data.info()

In [ ]:
# Printing the statistics
twitter_data.describe()

In [ ]:
# Checking the ditribution of traget column
twitter_data['target'].value_counts()

In [ ]:
# Counting the number of missing values in each column
twitter_data.isnull().sum()

**GRAPHS & CHARTS**

Sentiment Dsitribution Bar Chart (Shows Class balance)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.countplot(x='target', data=twitter_data, palette='viridis')
plt.title('Sentiment Class Distribution')
plt.xlabel('Sentiment')
plt.ylabel('Count')
plt.show()

0 --> Negative

1 --> Neutral

2 --> Positive

Tweet Length Distribution

In [ ]:
twitter_data['tweet_length'] = twitter_data['text'].apply(len)
twitter_data.head()

Tweet Length Distribution Histogram

In [ ]:
sns.histplot(twitter_data['tweet_length'], bins=50)
plt.title("Tweet Length Distribution")
plt.show()

Tweet Length Distribution Pie Chart

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Calculate tweet length
twitter_data['tweet_length'] = twitter_data['text'].apply(len)

# Define categories
bins = [0, 50, 100, 150, 200, 280]  # Twitter max length = 280 chars
labels = ['0-50', '51-100', '101-150', '151-200', '201-280']

# Categorize tweets
twitter_data['length_category'] = pd.cut(twitter_data['tweet_length'], bins=bins, labels=labels, right=True)

# Count category frequencies
length_counts = twitter_data['length_category'].value_counts().sort_index()

# Plot pie chart
plt.pie(length_counts, labels=length_counts.index, autopct='%1.1f%%', startangle=90, colors=plt.cm.Paired.colors)
plt.title('Tweet Length Distribution (Pie Chart)')
plt.axis('equal')  # Equal aspect ratio for perfect circle
plt.show()

Word Clouds - for each sentiment (positive & negative tweets)

In [ ]:
from wordcloud import WordCloud

pos_text = " ".join(twitter_data[twitter_data['target'] == 2]['text'])
neg_text = " ".join(twitter_data[twitter_data['target'] == 0]['text'])
neutral_text = " ".join(twitter_data[twitter_data['target'] == 1]['text'])

# Positive tweets word cloud
pos_wc = WordCloud(width=800, height=400).generate(pos_text)
plt.figure(figsize=(10, 5))
plt.imshow(pos_wc, interpolation='bilinear')
plt.axis("off")
plt.title("Positive Tweets Word Cloud")
plt.show()

# Negative tweets word cloud
neg_wc = WordCloud(width=800, height=400).generate(neg_text)
plt.figure(figsize=(10, 5))
plt.imshow(neg_wc, interpolation='bilinear')
plt.axis("off")
plt.title("Negative Tweets Word Cloud")
plt.show()

# Neutral tweets word cloud
neutral_wc = WordCloud(width=800, height=400).generate(neutral_text)
plt.figure(figsize=(10, 5))
plt.imshow(neutral_wc, interpolation='bilinear')
plt.axis("off")
plt.title("Neutral Tweets Word Cloud")
plt.show()


Most Frequently Used Words

In [ ]:
from collections import Counter
import pandas as pd

# Positive tweets
positive_words = " ".join(twitter_data[twitter_data['target']==2]['text']).split()
word_freq = Counter(positive_words)
common_words = pd.DataFrame(word_freq.most_common(20), columns=['Word', 'Frequency'])

sns.barplot(x='Frequency', y='Word', data=common_words, palette='crest')
plt.title('Top 20 Words in Positive Tweets')
plt.show()

# Negative Tweets
negative_words = " ".join(twitter_data[twitter_data['target']==0]['text']).split()
word_freq = Counter(negative_words)
common_words = pd.DataFrame(word_freq.most_common(20), columns=['Word', 'Frequency'])

sns.barplot(x='Frequency', y='Word', data=common_words, palette='crest')
plt.title('Top 20 Words in Negative Tweets')
plt.show()

# Neutral Tweets
neutral_words = " ".join(twitter_data[twitter_data['target']==1]['text']).split()
word_freq = Counter(neutral_words)
common_words = pd.DataFrame(word_freq.most_common(20), columns=['Word', 'Frequency'])

sns.barplot(x='Frequency', y='Word', data=common_words, palette='crest')
plt.title('Top 20 Words in Neutral Tweets')
plt.show()

**DATA PREPROCESSING**

Dropping the neutral rows

In [ ]:
# Drop rows where label == 1
twitter_data = twitter_data[twitter_data['target'] != 1].copy()

# Reset index after dropping
twitter_data = twitter_data.reset_index(drop=True)

print(twitter_data['target'].value_counts())
print(twitter_data.shape)

Converting the target label for positive tweets from '2' -> '1'

---



In [ ]:
 twitter_data.replace({'target':{2:1}},inplace=True)

In [ ]:
twitter_data.sample(5)

In [ ]:
# Checking the ditribution of traget column
twitter_data['target'].value_counts()

'0' --> Negative Tweet


'1' --> Positive Tweet

Dropping columns that are not required

In [ ]:
twitter_data.shape

Removing Dupliactes

In [ ]:
twitter_data = twitter_data.drop_duplicates(subset='text')

In [ ]:
twitter_data.shape

In [ ]:
# Checking the ditribution of traget column
twitter_data['target'].value_counts()

**DATA CLEANING**

**STEMMING** - Process of reducing a word into it's key/root word

Example - actor,actress,acting = act

&

**REMOVING NOISY TWEETS** - like the ones having emojis, urls, etc which do not affect the sentiment of the tweet

&

**REMOVAL OF STOPWORDS** - Removing words like 'a','an','my', etc. that do not directly affect the sentiment of the tweet

from nltk.stem.porter import PorterStemmer
import re

port_stem = PorterStemmer()

def stemming(content):    # we will pass the text column as content in this stemming function
  stemmed_content= re.sub('[^a-zA-Z]'," ", content)   # Removing everything that is not a lowercase or uppercase letter in the tweet (eg- @,;,etc.)
  stemmed_content= stemmed_content.lower()   # converting upper case letters to lower case letters
  stemmed_content= stemmed_content.split()   # split all the stemmed content words from the teweet and store them in a list
  stemmed_content= [port_stem.stem(word) for word in stemmed_content if not word in stopwords.words('english')]   # Performing stemming & Keeping only the words in the processed stem content that do not belong to the stopwords
  stemmed_content= ' '.join(stemmed_content)   # Joining all the words of stem content of a tweet into a single tweet

  return stemmed_content

- Applying the 'stemming' function to the dataset we have
- Creating a new column called 'stemmed_content'

twitter_data['stemmed_content']= twitter_data['text'].apply(stemming)

In [ ]:
'''
from nltk.stem.porter import PorterStemmer
import re

port_stem = PorterStemmer()
'''

In [ ]:
'''
def stemming(content):    # we will pass the text column as content in this stemming function
  stemmed_content= re.sub('[^a-zA-Z]'," ", content)   # Removing everything that is not a lowercase or uppercase letter in the tweet (eg- @,;,etc.)
  stemmed_content= stemmed_content.lower()   # converting upper case letters to lower case letters
  stemmed_content= stemmed_content.split()   # split all the stemmed content words from the teweet and store them in a list
  stemmed_content= [port_stem.stem(word) for word in stemmed_content if not word in stopwords.words('english')]   # Performing stemming & Keeping only the words in the processed stem content that do not belong to the stopwords
  stemmed_content= ' '.join(stemmed_content)   # Joining all the words of stem content of a tweet into a single tweet

  return stemmed_content
'''

In [ ]:
'''
# Applying the 'stemming' function to the dataset we have
# Creating a new column called 'stemmed_content'

twitter_data['stemmed_content']= twitter_data['text'].apply(stemming)
'''

**PREPROCESSING TECHNIQUES** - Lemmetization, Tokenization, Stopword Removal, Lowercasing, Removal of emojis, hashtags, punctuation, etc.

In [ ]:
# Downloading the stopwords
import nltk
nltk.download('stopwords')

# Printing the stopwords in English
print(stopwords.words('english'))

In [ ]:
# ADVANCED TEXT PREPROCESSING

# This step cleans the tweets more aggressively than before to reduce noise and improve accuracy.

import re
import string
import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_tweet(text):
    # Lowercase
    text = text.lower()
    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)
    # Remove usernames
    text = re.sub(r'@\w+', '', text)
    # Remove hashtags symbol (keep the text)
    text = re.sub(r'#', '', text)
    # Remove emojis and non-ASCII
    text = text.encode('ascii', 'ignore').decode('ascii')
    # Remove numbers
    text = re.sub(r'\d+', '', text)
    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    # Remove repeated characters (e.g., loooove -> love)
    text = re.sub(r'(.)\1{2,}', r'\1', text)
    # Tokenize and remove stopwords
    words = text.split()
    words = [w for w in words if w not in stop_words]
    # Lemmatize words
    words = [lemmatizer.lemmatize(w) for w in words]
    return " ".join(words)

# Apply cleaning to the dataset
twitter_data['clean_tweet'] = twitter_data['text'].apply(clean_tweet)
print("Sample cleaned tweets:", twitter_data['clean_tweet'].head())

In [ ]:
twitter_data.sample(5)

In [ ]:
twitter_data.shape

In [ ]:
print(twitter_data['clean_tweet'])

In [ ]:
# Separating the data (tweet/text) and label(target)
X= twitter_data['clean_tweet'].values
Y= twitter_data['target'].values

In [ ]:
print(X)

In [ ]:
print(Y)

**DATASET CLEANLINESS METRICS**

In [ ]:
# =====================================
# Dataset Cleanliness Metrics
# Run this cell after you create the 'clean_tweet' column in twitter_data
# =====================================

import re
import pandas as pd
import numpy as np
import nltk

# Try to use nltk words for OOV detection, fallback to stopwords if not available
try:
    from nltk.corpus import words
    english_vocab = set(words.words())
except LookupError:
    nltk.download('stopwords')
    from nltk.corpus import stopwords
    english_vocab = set(stopwords.words('english'))
    print("⚠️ Using stopwords list instead of full English vocab for OOV calculation")

def compute_cleanliness_metrics(original_col="text", cleaned_col="clean_tweet"):
    metrics = {}

    # Tweet lengths
    twitter_data["orig_len"] = twitter_data[original_col].str.len()
    twitter_data["clean_len"] = twitter_data[cleaned_col].str.len()

    # Reduction %
    twitter_data["reduction_pct"] = (twitter_data["orig_len"] - twitter_data["clean_len"]) / twitter_data["orig_len"] * 100

    metrics["Avg Length (Original)"] = twitter_data["orig_len"].mean()
    metrics["Avg Length (Cleaned)"] = twitter_data["clean_len"].mean()
    metrics["Std Dev (Original)"] = twitter_data["orig_len"].std()
    metrics["Std Dev (Cleaned)"] = twitter_data["clean_len"].std()
    metrics["% Characters Removed"] = twitter_data["reduction_pct"].mean()

    # Special tokens
    def count_specials(text):
        hashtags = len(re.findall(r"#\w+", text))
        mentions = len(re.findall(r"@\w+", text))
        urls = len(re.findall(r"http\S+", text))
        emojis = len(re.findall(r"[^\w\s,]", text))  # crude emoji/symbol count
        return hashtags, mentions, urls, emojis

    specials = twitter_data[original_col].apply(count_specials)
    twitter_data["hashtags"], twitter_data["mentions"], twitter_data["urls"], twitter_data["emojis"] = zip(*specials)

    metrics["Avg Hashtags"] = twitter_data["hashtags"].mean()
    metrics["Avg Mentions"] = twitter_data["mentions"].mean()
    metrics["Avg URLs"] = twitter_data["urls"].mean()
    metrics["Avg Emojis/Symbols"] = twitter_data["emojis"].mean()

    # Vocabulary size & OOV
    all_tokens = " ".join(twitter_data[cleaned_col]).split()
    vocab = set(all_tokens)
    metrics["Vocabulary Size"] = len(vocab)

    oov = [t for t in vocab if t.lower() not in english_vocab and t.isalpha()]
    metrics["OOV Rate (%)"] = len(oov) / len(vocab) * 100 if vocab else 0

    return pd.DataFrame(metrics, index=[0])

# 🔹 Run metrics on twitter_data
cleanliness_results = compute_cleanliness_metrics(original_col="text", cleaned_col="clean_tweet")
display(cleanliness_results)


**TRAIN-TEST SPLIT**

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, stratify=Y, random_state=42)

# X_train, X_test, Y_train, Y_test are the 4 arrays
# X_train contains all the training data tweets and their corresponding targets are stored in the Y_train
# X_test contains all the test data tweets and their corresponding targets are stored in the Y_test
# test_size= 0.2 --> means 20% of the data is test data and the rest 80% data is used for training the ML model
# stratify=Y --> means that in the training Y_train data & the testing Y_test data , I want an almost equal proportion of 0 and 1 from the target column
# random_state= 'any no' --> everyone using the same value of random_state will have the same rows in the test data and the train data

In [ ]:
print(X.shape, X_train.shape, X_test.shape)

In [ ]:
print(Y.shape, y_train.shape, y_test.shape)

In [ ]:
print(X_train)

In [ ]:
print(X_test)

# **FEATURE ENGINEERING**

**TF-IDF Vectorizer** --> used to convert textual/raw data into numerical data, that the model can understand, based on the weights given to each word or phrase depeding on it's frequency. Lesser the frequency of a word, more is the wight associated to it.

Then the ML model will associate the numerical data,i.e, weights associated to the words/phrases to the target labels(0 or 1) and work on it.

TF-IDF stands for Term Frequency – Inverse Document Frequency.

It assigns a weight to each word (or phrase) based on:

Term Frequency (TF) → How often the word appears in a tweet.

Inverse Document Frequency (IDF) → How rare the word is across all tweets.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()

In [ ]:
X_train= vectorizer.fit_transform(X_train)
X_test= vectorizer.transform(X_test)

In [ ]:
print(X_train)

In [ ]:
print(X_test)

All the words in each tweet get associated to a numerical value.

Example - here all the words in the tweet with 0 index are converted to numerical data

  (0, 222329)	0.45902566553974955
  (0, 286158)	0.36307656788596043
  (0, 358197)	0.30487157492746747
  (0, 408896)	0.663912210812278
  (0, 436771)	0.23863127215930252
  (0, 453351)	0.25845668749555184

BALANCING THE POSITIVE AND NEGATIVE CLASSES - SMOTE

In [ ]:
from imblearn.over_sampling import SMOTE
from collections import Counter

# Check original distribution
print("Original class distribution:", Counter(y_train))

# Apply SMOTE only on training data (never on test data!)
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

# Check new distribution
print("Resampled class distribution:", Counter(y_train_resampled))


**MODEL SELECTION AND TRAINING**

Writing code for all baseline ML models

Running models after SMOTE

In [ ]:
# ============================
# IMPORTS
# ============================
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from imblearn.over_sampling import SMOTE  # for balancing classes

# If XGBoost is installed
try:
    from xgboost import XGBClassifier
    xgb_available = True
except ImportError:
    xgb_available = False
    print("XGBoost not installed, skipping...")

# ============================
# APPLY SMOTE
# ============================
print("Original class distribution:", Counter(y_train))

smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print("Resampled class distribution:", Counter(y_train_resampled))

# ============================
# DEFINE MODELS (DEFAULT PARAMS)
# ============================
models = {
    "Logistic Regression": LogisticRegression(max_iter=2000),
    "Naive Bayes": MultinomialNB(),
    "Linear SVM": LinearSVC(),
    "KNN": KNeighborsClassifier(),
    "Random Forest": RandomForestClassifier(random_state=42),
}

if xgb_available:
    models["XGBoost"] = XGBClassifier(use_label_encoder=False, eval_metric="logloss")

# ============================
# TRAIN + EVALUATE MODELS
# ============================
for name, model in models.items():
    print("="*60)
    print(f"Training {name}...")

    model.fit(X_train_resampled, y_train_resampled)  # train on SMOTE data
    y_pred = model.predict(X_test)  # test on original data

    acc = accuracy_score(y_test, y_pred)
    print(f"{name} Test Accuracy: {acc:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=np.unique(y_test),
                yticklabels=np.unique(y_test))
    plt.title(f"{name} - Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.show()

# ============================
# ENSEMBLE VOTING
# ============================
estimators = [(name, model) for name, model in models.items()]
voting_clf = VotingClassifier(estimators=estimators, voting="hard")
voting_clf.fit(X_train_resampled, y_train_resampled)

y_pred = voting_clf.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print("="*60)
print("Ensemble Voting Classifier")
print(f"Accuracy: {acc:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=np.unique(y_test),
            yticklabels=np.unique(y_test))
plt.title("Ensemble Voting - Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()


Evaluation

In [ ]:
# ============================
# IMPORTS
# ============================
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score, roc_curve, auc

# ============================
# EVALUATE ALL MODELS AND STORE METRICS
# ============================
metrics_list = []

for name, model in models.items():
    model.fit(X_train_resampled, y_train_resampled)
    y_pred = model.predict(X_test)

    # For ROC-AUC, need probability scores or decision function
    if hasattr(model, "predict_proba"):
        y_score = model.predict_proba(X_test)[:,1]
    else:
        # Use decision function for models like LinearSVC
        y_score = model.decision_function(X_test)

    metrics_list.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "y_score": y_score  # store for ROC curve
    })

# Ensemble Voting
estimators = [(name, model) for name, model in models.items()]
voting_clf = VotingClassifier(estimators=estimators, voting="hard")
voting_clf.fit(X_train_resampled, y_train_resampled)
y_pred = voting_clf.predict(X_test)
# Use majority vote as probabilities for ROC-AUC approximation
y_score = voting_clf.predict_proba(X_test)[:,1] if hasattr(voting_clf, "predict_proba") else y_pred

metrics_list.append({
    "Model": "Voting Ensemble",
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred),
    "Recall": recall_score(y_test, y_pred),
    "F1": f1_score(y_test, y_pred),
    "y_score": y_score
})

# ============================
# CREATE DATAFRAME
# ============================
metrics_df = pd.DataFrame(metrics_list)
metrics_df_display = metrics_df.drop(columns=["y_score"])
print(metrics_df_display)

# ============================
# BAR CHART COMPARISON
# ============================
metrics_melted = metrics_df_display.melt(id_vars="Model", var_name="Metric", value_name="Score")
plt.figure(figsize=(12,6))
sns.barplot(data=metrics_melted, x="Model", y="Score", hue="Metric")
plt.title("Comparison of Models Across Metrics")
plt.xticks(rotation=45)
plt.ylim(0,1)
plt.legend(loc="lower right")
plt.show()

# ============================
# ROC CURVES
# ============================
plt.figure(figsize=(8,6))
for i, row in metrics_df.iterrows():
    fpr, tpr, _ = roc_curve(y_test, row["y_score"])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"{row['Model']} (AUC = {roc_auc:.3f})")

plt.plot([0,1],[0,1],'k--')  # diagonal
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves for All Models")
plt.legend(loc="lower right")
plt.show()


Applying Randomized Search CV to all models, except RandomForest

In [ ]:
# ============================
# IMPORTS
# ============================
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import RandomizedSearchCV

from imblearn.over_sampling import SMOTE  # for balancing classes

# If XGBoost is installed
try:
    from xgboost import XGBClassifier
    xgb_available = True
except ImportError:
    xgb_available = False
    print("XGBoost not installed, skipping...")

# ============================
# APPLY SMOTE
# ============================
print("Original class distribution:", Counter(y_train))

smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print("Resampled class distribution:", Counter(y_train_resampled))

# ============================
# DEFINE MODELS + PARAM GRIDS
# ============================
param_grids = {
    "Logistic Regression": {
        "C": np.logspace(-2, 2, 10),
        "solver": ["liblinear", "saga"]
    },
    "Naive Bayes": {
        "alpha": np.linspace(0.1, 1.0, 10)
    },
    "Linear SVM": {
        "C": np.logspace(-2, 2, 10)
    },
    "KNN": {
        "n_neighbors": [3, 5, 7, 9],
        "weights": ["uniform", "distance"]
    },
    # Random Forest is left untuned to save time
    "Decision Tree": {
        "criterion": ["gini", "entropy"],
        "max_depth": [10, 20, None],
        "min_samples_split": [2, 5, 10]
    }
}

if xgb_available:
    param_grids["XGBoost"] = {
        "n_estimators": [100, 200, 300],
        "max_depth": [4, 6, 8],
        "learning_rate": [0.01, 0.1, 0.2],
        "subsample": [0.8, 1.0],
        "colsample_bytree": [0.8, 1.0]
    }

base_models = {
    "Logistic Regression": LogisticRegression(max_iter=2000),
    "Naive Bayes": MultinomialNB(),
    "Linear SVM": LinearSVC(),
    "KNN": KNeighborsClassifier(),
    "Random Forest": RandomForestClassifier(random_state=42),  # not tuned
    "Decision Tree": DecisionTreeClassifier(random_state=42),
}

if xgb_available:
    base_models["XGBoost"] = XGBClassifier(use_label_encoder=False, eval_metric="logloss")

# ============================
# TRAIN + EVALUATE MODELS
# ============================
best_models = {}

for name, model in base_models.items():
    print("="*60)
    print(f"Tuning {name}...")

    param_grid = param_grids.get(name, {})
    if param_grid:  # tune if params exist
        search = RandomizedSearchCV(
            model,
            param_distributions=param_grid,
            n_iter=10,  # increased for better accuracy
            scoring="accuracy",
            cv=3,
            random_state=42,
            n_jobs=-1
        )
        search.fit(X_train_resampled, y_train_resampled)
        best_model = search.best_estimator_
        print(f"Best Params for {name}: {search.best_params_}")
    else:
        model.fit(X_train_resampled, y_train_resampled)
        best_model = model

    # Save best model
    best_models[name] = best_model

    # Evaluate
    y_pred = best_model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f"{name} Test Accuracy: {acc:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=np.unique(y_test),
                yticklabels=np.unique(y_test))
    plt.title(f"{name} - Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.show()

# ============================
# ENSEMBLE VOTING
# ============================
estimators = [(name, model) for name, model in best_models.items()]
voting_clf = VotingClassifier(estimators=estimators, voting="hard")
voting_clf.fit(X_train_resampled, y_train_resampled)

y_pred = voting_clf.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print("="*60)
print("Ensemble Voting Classifier")
print(f"Accuracy: {acc:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=np.unique(y_test),
            yticklabels=np.unique(y_test))
plt.title("Ensemble Voting - Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()


Applying GridSearchCV to all models, except RF

In [ ]:
# ============================
# IMPORTS
# ============================
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV

from imblearn.over_sampling import SMOTE

try:
    from xgboost import XGBClassifier
    xgb_available = True
except ImportError:
    xgb_available = False
    print("XGBoost not installed, skipping...")

# ============================
# APPLY SMOTE
# ============================
print("Original class distribution:", Counter(y_train))
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)
print("Resampled class distribution:", Counter(y_train_resampled))

# ============================
# BASE MODELS
# ============================
base_models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, solver="liblinear"),
    "Naive Bayes": MultinomialNB(),
    "Linear SVM": LinearSVC(max_iter=5000),
    "KNN": KNeighborsClassifier(),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42, n_estimators=200, max_depth=20),
}

if xgb_available:
    base_models["XGBoost"] = XGBClassifier(
        use_label_encoder=False, eval_metric="logloss", n_estimators=200, max_depth=6, learning_rate=0.1
    )

# ============================
# PARAM GRIDS (light + effective)
# ============================
param_grids = {
    "Logistic Regression": {
        "C": [0.5, 1, 2],
        "penalty": ["l1", "l2"],
        "solver": ["liblinear"]
    },
    "Naive Bayes": {
        "alpha": [0.5, 1.0]
    },
    "Linear SVM": {
        "C": [0.5, 1, 2]
    },
    "KNN": {
        "n_neighbors": [3, 5],
        "weights": ["uniform", "distance"]
    },
    "Decision Tree": {
        "criterion": ["gini", "entropy"],
        "max_depth": [10, 20]
    },
    # Random Forest is already tuned manually (skip GridSearch for runtime reasons)
}

if xgb_available:
    param_grids["XGBoost"] = {
        "n_estimators": [100, 200],
        "max_depth": [4, 6],
        "learning_rate": [0.05, 0.1],
        "subsample": [0.8, 1.0],
    }

# ============================
# GRID SEARCH
# ============================
best_models = {}

for name, model in base_models.items():
    print("="*60)
    print(f"Tuning {name}...")

    if name in param_grids:  # Apply GridSearchCV
        grid = GridSearchCV(model, param_grids[name], cv=2, n_jobs=-1, verbose=1)
        grid.fit(X_train_resampled, y_train_resampled)
        best_models[name] = grid.best_estimator_
        print(f"Best Params for {name}: {grid.best_params_}")
    else:  # Random Forest directly
        model.fit(X_train_resampled, y_train_resampled)
        best_models[name] = model

    # Evaluate
    y_pred = best_models[name].predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f"{name} Test Accuracy: {acc:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=np.unique(y_test),
                yticklabels=np.unique(y_test))
    plt.title(f"{name} - Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.show()

# ============================
# HARD VOTING ENSEMBLE
# ============================
estimators = [(name, model) for name, model in best_models.items()]
voting_clf = VotingClassifier(estimators=estimators, voting="hard")
voting_clf.fit(X_train_resampled, y_train_resampled)

y_pred = voting_clf.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print("="*60)
print("Hard Voting Ensemble Classifier")
print(f"Accuracy: {acc:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=np.unique(y_test),
            yticklabels=np.unique(y_test))
plt.title("Hard Voting Ensemble - Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()


Evaluating all baseline models + tuned LR & tuned XGBOOST

In [ ]:
# ============================
# IMPORTS
# ============================
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from collections import Counter

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    precision_recall_fscore_support, roc_curve, auc, roc_auc_score
)
from sklearn.model_selection import GridSearchCV
from sklearn.calibration import CalibratedClassifierCV

from imblearn.over_sampling import SMOTE

try:
    from xgboost import XGBClassifier
    xgb_available = True
except ImportError:
    xgb_available = False
    print("XGBoost not installed, skipping...")

# ============================
# APPLY SMOTE
# ============================
print("Original class distribution:", Counter(y_train))
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)
print("Resampled class distribution:", Counter(y_train_resampled))

# ============================
# BASE MODELS
# ============================
base_models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, solver="liblinear"),
    "Naive Bayes": MultinomialNB(),
    "Linear SVM": LinearSVC(max_iter=5000),
    "KNN": KNeighborsClassifier(),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42, n_estimators=200, max_depth=20),
}

if xgb_available:
    base_models["XGBoost"] = XGBClassifier(
        use_label_encoder=False,
        eval_metric="logloss",
        random_state=42
    )

# ============================
# PARAM GRIDS (only LR + XGB)
# ============================
param_grids = {
    "Logistic Regression": {
        "C": [0.5, 1, 2],
        "penalty": ["l1", "l2"],
        "solver": ["liblinear"]
    }
}

if xgb_available:
    param_grids["XGBoost"] = {
        "n_estimators": [100, 200],
        "max_depth": [4, 6],
        "learning_rate": [0.05, 0.1],
        "subsample": [0.8, 1.0],
    }

# ============================
# TRAIN + COLLECT METRICS
# ============================
metrics_list = {}
best_models = {}

for name, model in base_models.items():
    print("="*60)
    print(f"Training {name}...")

    if name in param_grids:  # tune LR + XGB
        grid = GridSearchCV(model, param_grids[name], cv=2, n_jobs=-1, verbose=0)
        grid.fit(X_train_resampled, y_train_resampled)
        best_models[name] = grid.best_estimator_
        print(f"Best Params for {name}: {grid.best_params_}")
    else:
        model.fit(X_train_resampled, y_train_resampled)
        best_models[name] = model

    # If model doesn't support predict_proba → wrap with calibration
    if hasattr(best_models[name], "predict_proba"):
        y_proba = best_models[name].predict_proba(X_test)[:, 1] if len(np.unique(y_test)) == 2 else None
    else:
        try:
            calibrated = CalibratedClassifierCV(best_models[name], cv=2)
            calibrated.fit(X_train_resampled, y_train_resampled)
            y_proba = calibrated.predict_proba(X_test)[:, 1] if len(np.unique(y_test)) == 2 else None
            best_models[name] = calibrated  # replace with calibrated
        except Exception:
            y_proba = None

    y_pred = best_models[name].predict(X_test)

    # Metrics
    acc = accuracy_score(y_test, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_test, y_pred, average="macro", zero_division=0
    )

    metrics_list[name] = {
        "Accuracy": acc,
        "Precision": precision,
        "Recall": recall,
        "F1-score": f1,
        "ROC-AUC": roc_auc_score(y_test, y_proba) if y_proba is not None else np.nan
    }

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=np.unique(y_test),
                yticklabels=np.unique(y_test))
    plt.title(f"{name} - Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.show()

# ============================
# HARD VOTING ENSEMBLE
# ============================
estimators = [(name, model) for name, model in best_models.items()]
voting_clf = VotingClassifier(estimators=estimators, voting="hard")
voting_clf.fit(X_train_resampled, y_train_resampled)

y_pred = voting_clf.predict(X_test)
acc = accuracy_score(y_test, y_pred)
precision, recall, f1, _ = precision_recall_fscore_support(
    y_test, y_pred, average="macro", zero_division=0
)

metrics_list["Hard Voting Ensemble"] = {
    "Accuracy": acc,
    "Precision": precision,
    "Recall": recall,
    "F1-score": f1,
    "ROC-AUC": np.nan  # Hard voting doesn't give probs
}

# ============================
# METRICS TABLE
# ============================
metrics_df = pd.DataFrame(metrics_list).T
print("\n=== Metrics Table ===")
print(metrics_df)

# ============================
# BAR CHART COMPARISON
# ============================
metrics_melted = metrics_df.reset_index().melt(id_vars="index", var_name="Metric", value_name="Score")
plt.figure(figsize=(12, 6))
sns.barplot(data=metrics_melted, x="index", y="Score", hue="Metric")
plt.xticks(rotation=45)
plt.title("Comparison of All Models (Accuracy, Precision, Recall, F1, ROC-AUC)")
plt.ylabel("Score")
plt.xlabel("Model")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.show()

# ============================
# ROC-AUC CURVES (only prob models)
# ============================
plt.figure(figsize=(8, 6))
for name, model in best_models.items():
    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X_test)[:, 1]
        fpr, tpr, _ = roc_curve(y_test, y_proba)
        plt.plot(fpr, tpr, label=f"{name} (AUC = {auc(fpr, tpr):.3f})")
plt.plot([0, 1], [0, 1], "k--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves for Models")
plt.legend(loc="lower right")
plt.show()


ROC-AUC for all baseline + Ensemble models

In [ ]:
# ============================
# ROC–AUC CURVES (ALL MODELS + ENSEMBLE)
# ============================
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize

# Ensure binary labels
y_test_bin = label_binarize(y_test, classes=[0,1]).ravel()

plt.figure(figsize=(10,7))

for name, model in best_models.items():
    # Not all models have predict_proba (like LinearSVC), so handle decision_function
    if hasattr(model, "predict_proba"):
        y_score = model.predict_proba(X_test)[:, 1]
    elif hasattr(model, "decision_function"):
        y_score = model.decision_function(X_test)
    else:
        continue  # skip models that can't give scores

    fpr, tpr, _ = roc_curve(y_test_bin, y_score)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"{name} (AUC = {roc_auc:.3f})")

# Add Hard Voting Ensemble
if hasattr(voting_clf, "predict_proba"):
    y_score_ensemble = voting_clf.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test_bin, y_score_ensemble)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"Hard Voting Ensemble (AUC = {roc_auc:.3f})")

# Plot settings
plt.plot([0,1], [0,1], "k--", lw=2)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC–AUC Curves for All Models")
plt.legend(loc="lower right")
plt.show()


Text vs Clean_tweet

In [ ]:
twitter_data.head()

Text length distribution of clean_tweets

In [ ]:
twitter_data['clean_tweet_length'] = twitter_data['clean_tweet'].apply(len)
twitter_data.head()

Histogram showing clean_tweet_length ditribution

In [ ]:
sns.histplot(twitter_data['clean_tweet_length'], bins=50, color='pink')
plt.title("Clean Tweet Length Distribution")
plt.show()

Visual Comparision between 'Original Tweet Length' and 'Clean Tweet Length'

In [ ]:
# Plot comparison
plt.figure(figsize=(10,6))
plt.hist(twitter_data['tweet_length'], bins=50, alpha=0.5, label='Original Tweets', color='cyan', edgecolor='black')
plt.hist(twitter_data['clean_tweet_length'], bins=50, alpha=0.5, label='Cleaned Tweets', color='magenta', edgecolor='black')

plt.title('Tweet Length Comparison: Original vs Cleaned')
plt.xlabel('Tweet Length (characters)')
plt.ylabel('Frequency')
plt.legend()
plt.show()

In [ ]:
twitter_data['target'].value_counts()

Word Cloud and most Used Words -- on Clean_tweets

In [ ]:
# Word Cloud

from wordcloud import WordCloud

neg_text = " ".join(twitter_data[twitter_data['target'] == 0]['clean_tweet'])
pos_text = " ".join(twitter_data[twitter_data['target'] == 1]['clean_tweet'])

# Positive tweets word cloud
pos_wc = WordCloud(width=800, height=400).generate(pos_text)
plt.figure(figsize=(10, 5))
plt.imshow(pos_wc, interpolation='bilinear')
plt.axis("off")
plt.title("Positive Tweets Word Cloud")
plt.show()

# Negative tweets word cloud
neg_wc = WordCloud(width=800, height=400).generate(neg_text)
plt.figure(figsize=(10, 5))
plt.imshow(neg_wc, interpolation='bilinear')
plt.axis("off")
plt.title("Negative Tweets Word Cloud")
plt.show()


In [ ]:
# Most used words

from collections import Counter
import pandas as pd

# Positive tweets
positive_words = " ".join(twitter_data[twitter_data['target']==1]['clean_tweet']).split()
word_freq = Counter(positive_words)
common_words = pd.DataFrame(word_freq.most_common(20), columns=['Word', 'Frequency'])

sns.barplot(x='Frequency', y='Word', data=common_words, palette='crest')
plt.title('Top 20 Words in Positive Tweets')
plt.show()

# Negative Tweets
negative_words = " ".join(twitter_data[twitter_data['target']==0]['clean_tweet']).split()
word_freq = Counter(negative_words)
common_words = pd.DataFrame(word_freq.most_common(20), columns=['Word', 'Frequency'])

sns.barplot(x='Frequency', y='Word', data=common_words, palette='crest')
plt.title('Top 20 Words in Negative Tweets')
plt.show()
